# Data Ingestion into Bronze Layer
1. Get the list of data to be ingested into bronze layer from metadata tables table
2. loop through each record in the metadata table and using the metadata details ingest the data

In [0]:
#Imports
from pyspark.sql.functions import current_timestamp, col
import requests
import json

In [0]:
catalog_name = "neo_bank"

In [0]:
secret_scope_name = "neobank-scope"
secret_key_name = "neobank-jdbc-connection-json"

secret_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key=secret_key_name
    )

config = json.loads(secret_json)

# Azure SQL Database connection parameters
jdbc_hostname = config["jdbc_hostname"]
jdbc_port = config["jdbc_port"]
jdbc_database = config["jdbc_database"]
jdbc_username = config["jdbc_username"]
jdbc_password = config["jdbc_password"] 
driver = config["driver"]

# Construct JDBC URL
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=60;socketTimeout=300000;"

# Connection properties
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": driver
}

In [0]:
tables_df = spark.read.table(f"{catalog_name}.metadata.tables")
filtered_df = tables_df.filter(col("active_flag")=="true")

In [0]:
for row in filtered_df.collect():
    table_id = row["table_id"]
    table_name = row["table_name"]
    source_system = row["source_system"]
    source_schema = row["source_schema"]
    source_table = row["source_table"]
    source_path = row["source_path"]
    bronze_schema = row["bronze_schema"]

    #getting table parameters with table_id
    table_parameters_df = spark.read.table(f"{catalog_name}.metadata.table_parameters")
    table_parameters_filtered_df=table_parameters_df.filter(col("table_id")==table_id) 

    load_type_row = table_parameters_filtered_df.filter(col("parameter_name")=="load_type").select(col("parameter_value")).first()
    load_type = load_type_row[0] if load_type_row else None

    primary_key_row = table_parameters_filtered_df.filter(col("parameter_name")=="primary_key").select(col("parameter_value")).first()
    primary_key = primary_key_row[0] if primary_key_row else None

    watermark_column_row = table_parameters_filtered_df.filter(col("parameter_name")=="watermark_column").select(col("parameter_value")).first()
    watermark_column = watermark_column_row[0] if watermark_column_row else None


    #getting the last watermark value for that table with table_id
    watermark_df = spark.read.table(f"{catalog_name}.metadata.table_watermarks")
    watermark_row = watermark_df.filter(col("table_id")==table_id).select(col("last_watermark_value")).first()
    last_watermark = watermark_row[0] if watermark_row else None

    if(source_system=="neo_bank_db"):

        final_source_table = f"{source_schema}.{source_table}"
        final_target_table = f"{catalog_name}.{bronze_schema}.{table_name}"


        #build query to retrieve the data
        if load_type in ["APPEND", "MERGE"]:
            query = f"(select * from {source_schema}.{source_table} where {watermark_column}>'{last_watermark}') src"
        else:
            query = f"(select * from {source_schema}.{source_table}) src"


        # Read data from Azure SQL Database
        df = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=connection_properties
        )

        df=df.withColumn("insert_timestamp",current_timestamp())


        if load_type in ["APPEND", "MERGE"]:
        # Using append mode to add data incrementally
            (   
                df.write 
                .format("delta") 
                .mode("append") 
                .option("overwriteSchema", "true")
                .saveAsTable(final_target_table)
            )
        
        if load_type in ["FULL"]:
            (   
                df.write 
                .format("delta") 
                .mode("overwrite") 
                .option("overwriteSchema", "true")
                .saveAsTable(final_target_table)
            )


    elif(source_system=="volumes"):

        df = (
                spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format","csv")
                    .option("cloudFiles.schemaLocation",f"/Volumes/neo_bank/landing/files/_schema/{table_name}")
                    .option("header","True")
                    .load(source_path)
            )

        df=df.withColumn("insert_timestamp",current_timestamp())

        (
            df.writeStream
                .format("delta")
                .option("checkpointLocation",f"/Volumes/neo_bank/landing/files/_checkpoints/{table_name}")
                .outputMode("append")
                .trigger(availableNow=True)
                .toTable(f"{catalog_name}.{bronze_schema}.{table_name}")
        )